# 01 — Exploratory data analysis

PhysioNet 2019 Sepsis Challenge: per-hour vitals + labs across two hospitals.
We focus on the three things that drive the modelling decisions downstream:

1. **Class prevalence and time-to-onset** — how rare is sepsis, and how late in a stay does it land?
2. **Missingness pattern** — which variables are charted hourly vs only on lab draws?
3. **Hospital differences** — A vs B prevalence and feature distributions, the basis for the cross-hospital protocol in notebook 04.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import load_dataset, split_by_hospital, FEATURES, VITALS, LABS, TARGET
from src.visualize import plot_missingness_heatmap, plot_time_to_sepsis
sns.set_theme(style='whitegrid')
DATA_ROOT = '../data'

In [ ]:
records = load_dataset(DATA_ROOT, hospitals=('A','B'), subset=5000, seed=0)
print(f'{len(records)} patients loaded')
by_h = split_by_hospital(records)
for h, recs in by_h.items():
    sep = sum(1 for r in recs if r.is_sepsis)
    print(f'  Hospital {h}: {len(recs)} patients, {sep} septic ({sep/len(recs):.2%})')

## Time-to-onset distribution

The label flips to 1 at `t_sepsis - 6`. Onset hours cluster early in the stay (most sepsis is identified within the first 1–2 days), with a long tail of late-onset cases.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
plot_time_to_sepsis(records, ax=ax); plt.show()

## Missingness over time

Vitals (HR, MAP, Temp, SpO2, RR) are charted ~hourly. Labs (lactate, WBC, creatinine, ...) are sparse — often only a handful of measurements over a multi-day stay. The mask channel and the time-since-last-measurement channel are therefore part of the model input, not just the imputed value.

In [ ]:
from src.features import featurize_many
frame = featurize_many(records[:1500])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_missingness_heatmap(frame, VITALS, ax=axes[0])
axes[0].set_title('Vitals — missingness')
plot_missingness_heatmap(frame, LABS[:14], ax=axes[1])
axes[1].set_title('Labs — missingness')
plt.tight_layout(); plt.show()

## Hospital-level comparisons

A vs B differ in case-mix and charting cadence. These distributional differences are exactly the source of the domain shift we quantify in notebook 04.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['HR', 'MAP', 'Lactate']):
    for h, recs in by_h.items():
        vals = pd.concat([r.df[col] for r in recs]).dropna()
        sns.kdeplot(vals, ax=ax, label=f'Hosp {h} (n={len(vals):,})')
    ax.set_title(col); ax.legend(frameon=False)
plt.tight_layout(); plt.show()